# Memory Mosaics vs GPT-2: Fig. 7, one block

Trains both 1-block models with the paper's model, data and optimizer, and plots training and validation
loss as in the 1-block panel of Fig. 7 ([arXiv:2405.06394v3](https://arxiv.org/abs/2405.06394v3)).

The default quick run takes ~15 min on a Colab TPU or GPU (fastest: v6e-1 TPU): sanity checks, then batch 32
and as many steps as fit `TRAIN_MINUTES`. That is the start of training (the paper's curves begin after ~1B tokens), so compare
trends, not values. `PAPER_RUN = True` uses the full recipe: 21B tokens per model.

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Transparent hugepages")  # harmless TPU startup notice
!pip install -q tiktoken py7zr multivolumefile
!test -d memory-mosaics-jax || git clone -q https://github.com/rohitium/memory-mosaics-jax.git
import sys
sys.path.insert(0, "memory-mosaics-jax/src")
import jax
print(jax.devices())

In [ ]:
import dataclasses, time
import numpy as np
from mm_jax import config, gpt2, memory_mosaic, train
from mm_jax.data import batches, prepare_babistories

PAPER_RUN = False
TRAIN_MINUTES = 8  # quick run: training time for both models together (~15 min in all)

models = {"GPT-2": (gpt2, config.ModelConfig()), "Memory Mosaic": (memory_mosaic, config.MosaicConfig())}
big_memory = (jax.devices()[0].memory_stats() or {}).get("bytes_limit", 0) > 30e9  # A100, v6e
t_cfg = config.TrainConfig() if PAPER_RUN else config.TrainConfig(batch_size=32, micro_batch_size=32 if big_memory else 8)
T = config.ModelConfig().block_size
bins = prepare_babistories("/content/babistories")
train_ids, val_ids = (np.memmap(bins[s], np.uint16, mode="r") for s in ("train", "val"))

In [ ]:
# Sanity checks: smoke tests, the paper's token counts, readable data, initial loss ≈ ln(vocab)
import tiktoken
sys.path.insert(0, "memory-mosaics-jax/tests")
import test_smoke
with jax.default_matmul_precision("highest"):  # tests compare against exact references
    for name in [n for n in dir(test_smoke) if n.startswith("test_")]:
        getattr(test_smoke, name)()
        print("ok", name)
assert (len(train_ids), len(val_ids)) == (474_704_907, 4_749_107)  # paper, Table 1
print(repr(tiktoken.get_encoding("gpt2").decode(val_ids[:40].tolist())))
x, y = next(batches(val_ids, 8, T))
for name, (m, cfg) in models.items():
    loss = float(m.apply(m.init(jax.random.key(0), cfg), x, cfg, y)[1])
    print(f"{name}: initial loss {loss:.2f}, ln(vocab) = {np.log(cfg.vocab_size):.2f}")
    assert abs(loss - np.log(cfg.vocab_size)) < 0.5

In [ ]:
def seconds_per_step(init_state, train_step, _):
    state, batch = init_state(jax.random.key(0)), jax.device_put(next(batches(train_ids, t_cfg.batch_size, T)))
    for i in range(7):  # 2 calls to compile, then 5 timed
        if i == 2:
            loss.block_until_ready()
            t0 = time.time()
        state, loss = train_step(state, batch, 0.0)
    loss.block_until_ready()
    return (time.time() - t0) / 5

trainers = {name: train.make_trainer(m, cfg, t_cfg) for name, (m, cfg) in models.items()}
if not PAPER_RUN:
    steps = int(TRAIN_MINUTES * 60 / 1.1 / sum(seconds_per_step(*t) for t in trainers.values()))  # 10% for evaluation
    t_cfg = dataclasses.replace(t_cfg, max_steps=steps, warmup_steps=steps // 20, eval_every=max(1, steps // 8), eval_batches=8)
print(f"{t_cfg.max_steps:,} steps = {t_cfg.max_steps * t_cfg.batch_size * T / 1e6:,.0f}M tokens per model")

In [ ]:
def run(name):
    init_state, train_step, eval_step = trainers[name]
    state = init_state(jax.random.key(0))
    data = map(jax.device_put, batches(train_ids, t_cfg.batch_size, T))  # copies overlap compute
    val = batches(val_ids, t_cfg.micro_batch_size, T, seed=1)
    losses, val_loss = [], {}
    for step in range(1, t_cfg.max_steps + 1):
        state, loss = train_step(state, next(data), train.learning_rate(step - 1, t_cfg))
        losses.append(loss)  # stays on device: reading it every step would stall the loop
        if step % t_cfg.eval_every == 0 or step == t_cfg.max_steps:
            val_loss[step] = float(np.mean([eval_step(state["params"], next(val)) for _ in range(t_cfg.eval_batches)]))
            print(name, step, f"val {val_loss[step]:.3f}")
    return np.array(losses), val_loss

results = {name: run(name) for name in trainers}

In [ ]:
import matplotlib.pyplot as plt

W = max(1, t_cfg.max_steps // 20)  # smoothing window
for (name, (losses, val_loss)), c in zip(results.items(), "br"):
    plt.plot(np.arange(W, len(losses) + 1), np.convolve(losses, np.ones(W) / W, "valid"), c, label=f"{name} train")
    plt.plot(list(val_loss), list(val_loss.values()), c + "--o", label=f"{name} val")
plt.xlabel(f"iterations (batch {t_cfg.batch_size})")
plt.ylabel("cross-entropy loss")
plt.legend()
plt.savefig("fig7_1block.png", dpi=150)